In [ ]:
import cv2
import mediapipe as mp
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# ==========================================
# 1. CONFIGURAR PYTORCH
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18()
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 29)

model.load_state_dict(
    torch.load(
        "signnet_final.pth",
        map_location=device,
        weights_only=True
    )
)
model.to(device)
model.eval()

transform_pipeline = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

clases_letras = ['A', 'B', 'C', 'D', 'DEL', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'NOTHING', 'O', 'P', 'Q', 'R', 'S', 'SPACE', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']

# ==========================================
# 2. CONFIGURAR MEDIAPIPE TASKS
# ==========================================

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    num_hands=1,
    running_mode=VisionRunningMode.IMAGE
)

detector = HandLandmarker.create_from_options(options)

# ==========================================
# 3. WEBCAM
# ==========================================

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # CORRECCIÓN 1: El volteo horizontal (1) va DENTRO del ciclo para que afecte a cada frame
    frame = cv2.flip(frame, 1) 

    h, w, _ = frame.shape
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame_rgb
    )

    result = detector.detect(mp_image)
    letra_detectada = "No se detecta mano"

    if len(result.hand_landmarks) > 0:
        # CORRECCIÓN 3: Procesamos solo la primera mano detectada para evitar sobreescrituras
        hand_landmarks = result.hand_landmarks[0] 

        # Optimizamos inicializando los límites del Bounding Box
        x_min, y_min = w, h
        x_max, y_max = 0, 0

        # CORRECCIÓN 2: Unificamos el dibujado de puntos y el cálculo del Bounding Box en un solo ciclo
        for lm in hand_landmarks:
            cx = int(lm.x * w)
            cy = int(lm.y * h)

            # Dibujar landmark en el frame
            cv2.circle(frame, (cx, cy), 3, (0, 255, 0), -1)

            # Actualizar límites de la caja
            x_min = min(x_min, cx)
            y_min = min(y_min, cy)
            x_max = max(x_max, cx)
            y_max = max(y_max, cy)

        # Configurar margen (Offset)
        offset = 60
        xmin = max(0, x_min - offset)
        ymin = max(0, y_min - offset)
        xmax = min(w, x_max + offset)
        ymax = min(h, y_max + offset)

        # Dibujar el rectángulo
        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)

        # Recortar región de interés usando la imagen RGB
        mano_recortada = frame_rgb[ymin:ymax, xmin:xmax]

        if mano_recortada.size > 0:
            imagen_pil = Image.fromarray(mano_recortada)
            tensor_img = transform_pipeline(imagen_pil).unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = model(tensor_img)
                pred = torch.argmax(outputs, dim=1)
                letra_detectada = clases_letras[pred.item()]

    # Mostrar la interfaz en pantalla
    cv2.putText(
        frame,
        f"Letra: {letra_detectada}",
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (255, 0, 0),
        3
    )

    cv2.imshow("Detector de Lenguaje de Senas", frame)

    tecla = cv2.waitKey(1) & 0xFF
    if tecla == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
detector.close()

I0000 00:00:1786423825.693057   21393 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1786423825.696858   21411 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.6), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786423825.722711   21396 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786423825.744842   21400 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786423827.425371   21397 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
QFontDatabase: Cannot find font directory /home/fred/miniconda3/envs/Deus/lib/python3.10/site-packages/cv2/qt/fo

In [1]:
pwd

'/home/fred/asl/code'